# 04 — Finetune CSM-1B (voz do Pedro) via Unsloth · T4 grátis

LoRA sobre `unsloth/csm-1b` (mirror Apache sem gate) com o dataset gravado pelo kit.
Receita verificada em 2026-06-10 contra o notebook oficial `Sesame_CSM_(1B)-TTS.ipynb`
do Unsloth + doc HF do CSM (ver `research/dossier-2026-06/10-sesame-csm.md`).

**Pré-requisito:** rodou o notebook 01 (dataset_v1 no Drive). **GPU:** T4 basta (L4 = folga).

Pegadinhas cobertas: voz varia sem contexto (sempre gerar com referência);
`max_new_tokens=125`≈10s (aumentamos); `audio max_length` calculado do NOSSO dataset.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
GH_TOKEN = userdata.get('GH_TOKEN')
!git clone https://{GH_TOKEN}@github.com/pedrocormann/TTS-ptbr.git /content/TTS-ptbr 2>/dev/null || (cd /content/TTS-ptbr && git pull)
%cd /content/TTS-ptbr
!mkdir -p data && ln -sfn /content/drive/MyDrive/TTS-ptbr-data/dataset_v1 data/dataset_v1
!ls data/dataset_v1/

In [ ]:
%%capture
# pins do notebook oficial Unsloth (2026-06): transformers==4.52.3, trl==0.22.2
import os, re
import torch
v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.52.3
!pip install --no-deps trl==0.22.2
!pip install torchcodec "datasets>=3.4.1,<4.0.0" soundfile jiwer librosa

## 1. Dataset → formato CSM (conversa speaker 0, áudio 24 kHz, tags inline)

In [ ]:
import json, pathlib
from datasets import Dataset, Audio

ROOT = pathlib.Path('data/dataset_v1')
rows = [json.loads(l) for l in (ROOT/'orpheus_train.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
# orpheus_*.jsonl já traz o texto com prefixo de estilo/sotaque (<animado|...>) — tags inline
raw_ds = Dataset.from_list([{'text': r['text'], 'audio': str(ROOT/r['audio'])} for r in rows])
raw_ds = raw_ds.cast_column('audio', Audio(sampling_rate=24000))
print(raw_ds)

# max_length de áudio calculado do NOSSO dataset (oficial usa 240001 = 10s fixos)
max_audio = max(len(ex['audio']['array']) for ex in raw_ds)
MAX_AUDIO = int(max_audio) + 1
MAX_TEXT = 384
print(f'max áudio: {max_audio/24000:.1f}s → max_length={MAX_AUDIO}')

In [ ]:
from unsloth import FastModel
from transformers import CsmForConditionalGeneration, AutoProcessor
import torch

model, processor = FastModel.from_pretrained(
    model_name = 'unsloth/csm-1b',      # mirror Apache-2.0, sem gate
    max_seq_length = 2048,
    dtype = None,
    auto_model = CsmForConditionalGeneration,
    load_in_4bit = False,               # 4-bit degrada aprendizado de áudio (guia Unsloth)
)

def preprocess(example):
    conversation = [{
        'role': '0',                    # speaker id 0 = Pedro
        'content': [
            {'type': 'text',  'text': example['text']},
            {'type': 'audio', 'path': example['audio']['array']},
        ],
    }]
    out = processor.apply_chat_template(
        conversation, tokenize=True, return_dict=True, output_labels=True,
        text_kwargs  = {'padding': 'max_length', 'max_length': MAX_TEXT,
                        'pad_to_multiple_of': 8, 'padding_side': 'right'},
        audio_kwargs = {'sampling_rate': 24000, 'max_length': MAX_AUDIO, 'padding': 'max_length'},
        common_kwargs = {'return_tensors': 'pt'},
    )
    return {k: v[0] for k, v in out.items()}

processed_ds = raw_ds.map(preprocess, remove_columns=raw_ds.column_names)
print(processed_ds)

## 2. LoRA + treino (r=64 — voz é "hero"; comunidade TTS usa rank > texto)

In [ ]:
model = FastModel.get_peft_model(
    model,
    r = 64, lora_alpha = 64, lora_dropout = 0, bias = 'none',
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407, use_rslora = False, loftq_config = None,
)

from transformers import TrainingArguments, Trainer
from unsloth import is_bfloat16_supported

trainer = Trainer(
    model = model, train_dataset = processed_ds,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,            # dataset pequeno: 2-3 épocas; vigie a loss
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(), bf16 = is_bfloat16_supported(),
        logging_steps = 5, optim = 'adamw_8bit',
        weight_decay = 0.001,
        lr_scheduler_type = 'linear', seed = 3407,
        output_dir = 'outputs', report_to = 'none',
        save_steps = 200, save_total_limit = 2,   # checkpoints (sessão Colab pode cair)
    ),
)
trainer.train()

## 3. Geração com CONTEXTO de voz (obrigatório — sem contexto a voz varia)
Gera o benchmark congelado `eval/benchmark_ptbr.jsonl` com 1 utterance de referência.

In [ ]:
import soundfile as sf, json, pathlib

ref = raw_ds[0]                                   # utterance de referência (limpa, neutra)
bench = [json.loads(l) for l in open('eval/benchmark_ptbr.jsonl', encoding='utf-8') if l.strip()]
outdir = pathlib.Path('gen_csm'); outdir.mkdir(exist_ok=True)

model.eval()
for i, item in enumerate(bench):
    conversation = [
        {'role': '0', 'content': [{'type': 'text', 'text': ref['text']},
                                  {'type': 'audio', 'path': ref['audio']['array']}]},
        {'role': '0', 'content': [{'type': 'text', 'text': item['text']}]},
    ]
    inputs = processor.apply_chat_template(conversation, tokenize=True, return_dict=True)
    with torch.no_grad():
        audio = model.generate(**inputs.to('cuda'),
                               max_new_tokens = 375,    # ~30s de teto (125≈10s)
                               depth_decoder_do_sample=True, depth_decoder_temperature=0.9,
                               do_sample=True, temperature=0.9,
                               output_audio = True)
    wav = audio[0].to(torch.float32).cpu().numpy()
    sf.write(outdir / f"{item.get('id', i):0>3}.wav", wav, 24000)
    print(item['text'][:60])

## 4. Eval (gates F1 do REPLAN: spk-sim ≥ 0.70 · WER ≤ 1.2× real · escuta)

In [ ]:
!pip -q install faster-whisper==1.1.0
!python -m eval.wer_roundtrip --in-dir gen_csm --transcripts eval/benchmark_ptbr.jsonl --model medium --lang pt
# referência da voz real = clipes neutros do dataset
!mkdir -p ref_pedro && python -c "
import json, shutil, pathlib
rows=[json.loads(l) for l in open('data/dataset_v1/train.jsonl', encoding='utf-8') if l.strip()]
neutros=[r for r in rows if r.get('style')=='neutro'][:20]
[shutil.copy(pathlib.Path('data/dataset_v1')/r['audio'], 'ref_pedro/') for r in neutros]"
!python -m eval.speaker_sim --ref-dir ref_pedro --gen-dir gen_csm

## 5. Salvar (adapter + merge) no Drive — o "ouro" fica fora do git

In [ ]:
SAVE = '/content/drive/MyDrive/TTS-ptbr-data/checkpoints/csm_lora_v1'
model.save_pretrained(SAVE); processor.save_pretrained(SAVE)
# merge fp16 opcional (p/ servir sem peft):
# model.save_pretrained_merged(SAVE + '_merged', processor, save_method='merged_16bit')
print('✅ salvo em', SAVE)
# reload em sessão nova:
# model, processor = FastModel.from_pretrained(model_name=SAVE, auto_model=CsmForConditionalGeneration, max_seq_length=2048)